In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Sector-wise Pairs Selection (Top-N per sector + Corr screen + Cointegration ranking)

Inputs:
- universe.csv with columns:
    ticker,sector
  Optional:
    free_float_mcap   (numeric; preferred for "top" selection if you have it)

Example universe.csv:
ticker,sector,free_float_mcap
TATAMOTORS.NS,Auto,1234567890
MOTHERSON.NS,Auto,987654321
HDFCBANK.NS,Banks,5555555555

Install:
pip install pandas numpy yfinance statsmodels scipy

Run:
python sector_pairs_selector.py

Outputs:
- sector_best_pairs.csv
- sector_all_candidates.csv
"""

import warnings
warnings.filterwarnings("ignore")

import math
import itertools
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

In [5]:
# =============================================================================
# CONFIG
# =============================================================================
@dataclass
class Config:
    UNIVERSE_CSV: str = "universe.csv"

    START: str = "2018-01-01"
    END: str = "2025-01-01"

    TOP_N_PER_SECTOR: int = 10

    # Choose how to define "top"
    # "free_float_mcap" -> uses free_float_mcap column in universe.csv (best if you have it)
    # "yfinance_mcap"   -> uses yfinance ticker.info['marketCap'] (fallback)
    # "liquidity"       -> uses avg(close*volume) over LIQ_LOOKBACK_DAYS (fallback)
    TOP_METHOD: str = "free_float_mcap"

    LIQ_LOOKBACK_DAYS: int = 252  # used only if TOP_METHOD="liquidity"

    # Correlation screening on log-prices
    CORR_LOOKBACK_DAYS: int = 252
    CORR_THRESHOLD: float = 0.75
    MAX_CORR_PAIRS_PER_SECTOR: int = 30  # after sorting by corr desc

    # Cointegration filter
    ADF_PVALUE_MAX: float = 0.05

    # Ranking weights (tweak as you like)
    W_CORR: float = 1.0
    W_LOGP: float = 0.25        # weight for -log10(pval)
    W_HALF_LIFE: float = 0.50   # weight for (1/(half_life+1))

    # Data field
    AUTO_ADJUST: bool = True    # yfinance: True => adjusted prices in "Close"


CFG = Config()


# =============================================================================
# HELPERS
# =============================================================================
def _safe_log(x: float, eps: float = 1e-300) -> float:
    return math.log(max(x, eps))


def fetch_prices_and_volume(tickers: List[str], start: str, end: str, auto_adjust: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (close_df, volume_df) aligned by date.
    """
    if not tickers:
        return pd.DataFrame(), pd.DataFrame()

    data = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=auto_adjust,
        group_by="column",
        threads=True,
        progress=False,
    )

    # yfinance returns different shapes depending on number of tickers
    # MultiIndex columns: ("Close", "TICKER"), ("Volume", "TICKER")
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"].copy() if "Close" in data.columns.levels[0] else pd.DataFrame()
        vol = data["Volume"].copy() if "Volume" in data.columns.levels[0] else pd.DataFrame()
    else:
        # single ticker case
        close = data[["Close"]].copy()
        vol = data[["Volume"]].copy()

    close = close.dropna(how="all")
    vol = vol.reindex(close.index).dropna(how="all")

    # Ensure columns are tickers (not multiindex)
    if isinstance(close.columns, pd.MultiIndex):
        close.columns = close.columns.get_level_values(-1)
    if isinstance(vol.columns, pd.MultiIndex):
        vol.columns = vol.columns.get_level_values(-1)

    return close, vol


def get_market_caps_yf(tickers: List[str]) -> Dict[str, Optional[float]]:
    """
    Slow-ish (loops tickers). Use only if you don't have free-float mcap in your CSV.
    """
    out = {}
    for t in tickers:
        try:
            info = yf.Ticker(t).info
            out[t] = float(info.get("marketCap")) if info.get("marketCap") is not None else None
        except Exception:
            out[t] = None
    return out


def pick_top_per_sector(universe: pd.DataFrame, cfg: Config) -> Dict[str, List[str]]:
    """
    Returns {sector: [tickers...]} selected top-N by cfg.TOP_METHOD.
    """
    universe = universe.copy()
    universe["ticker"] = universe["ticker"].astype(str).str.strip()
    universe["sector"] = universe["sector"].astype(str).str.strip()

    sector_map: Dict[str, List[str]] = {}
    for sec, g in universe.groupby("sector"):
        sector_map[sec] = g["ticker"].tolist()

    top: Dict[str, List[str]] = {}

    method = cfg.TOP_METHOD.lower().strip()

    if method == "free_float_mcap":
        if "free_float_mcap" not in universe.columns:
            print("[WARN] TOP_METHOD=free_float_mcap but column not found. Falling back to yfinance_mcap.")
            method = "yfinance_mcap"

    if method == "free_float_mcap":
        for sec, g in universe.groupby("sector"):
            gg = g.copy()
            gg["free_float_mcap"] = pd.to_numeric(gg["free_float_mcap"], errors="coerce")
            gg = gg.dropna(subset=["free_float_mcap"]).sort_values("free_float_mcap", ascending=False)
            top[sec] = gg["ticker"].head(cfg.TOP_N_PER_SECTOR).tolist()

    elif method == "yfinance_mcap":
        # Fetch caps for all tickers once
        all_tickers = sorted(universe["ticker"].unique().tolist())
        caps = get_market_caps_yf(all_tickers)

        for sec, ticks in sector_map.items():
            rows = [(t, caps.get(t)) for t in ticks]
            rows = [(t, c) for t, c in rows if c is not None]
            rows.sort(key=lambda x: x[1], reverse=True)
            top[sec] = [t for t, _ in rows[: cfg.TOP_N_PER_SECTOR]]

    elif method == "liquidity":
        # Liquidity = average dollar volume over last LIQ_LOOKBACK_DAYS
        all_tickers = sorted(universe["ticker"].unique().tolist())
        close, vol = fetch_prices_and_volume(all_tickers, cfg.START, cfg.END, auto_adjust=cfg.AUTO_ADJUST)
        if close.empty or vol.empty:
            raise RuntimeError("Could not download prices/volume for liquidity ranking.")

        # Use last LIQ_LOOKBACK_DAYS
        close2 = close.tail(cfg.LIQ_LOOKBACK_DAYS)
        vol2 = vol.reindex(close2.index).fillna(0.0)
        dollar_vol = (close2 * vol2).mean(axis=0, skipna=True)

        for sec, ticks in sector_map.items():
            dv = dollar_vol.reindex(ticks).dropna().sort_values(ascending=False)
            top[sec] = dv.index.tolist()[: cfg.TOP_N_PER_SECTOR]

    else:
        raise ValueError(f"Unknown TOP_METHOD: {cfg.TOP_METHOD}")

    # drop empty sectors
    top = {k: v for k, v in top.items() if len(v) >= 2}
    return top


def rolling_corr_pairs(log_prices: pd.DataFrame, lookback: int, corr_thresh: float, max_pairs: int) -> List[Tuple[str, str, float]]:
    """
    Returns list of (a, b, corr) sorted by corr desc.
    """
    if log_prices.shape[1] < 2:
        return []

    lp = log_prices.dropna(how="all")
    lp = lp.tail(lookback).dropna(axis=1, how="any")
    if lp.shape[1] < 2:
        return []

    corr = lp.corr()
    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a, b = cols[i], cols[j]
            c = float(corr.loc[a, b])
            if np.isfinite(c) and c >= corr_thresh:
                pairs.append((a, b, c))

    pairs.sort(key=lambda x: x[2], reverse=True)
    return pairs[:max_pairs]


def engle_granger_ols_adf(log_y: pd.Series, log_x: pd.Series) -> Tuple[float, float, pd.Series]:
    """
    OLS: log_y = a + b*log_x + e
    Returns (beta, adf_pvalue, residual_series)
    """
    df = pd.concat([log_y.rename("y"), log_x.rename("x")], axis=1).dropna()
    if len(df) < 50:
        return np.nan, np.nan, pd.Series(dtype=float)

    X = sm.add_constant(df["x"].values)
    model = sm.OLS(df["y"].values, X).fit()
    a, b = model.params[0], model.params[1]
    resid = df["y"] - (a + b * df["x"])

    try:
        pval = adfuller(resid.values, autolag="AIC")[1]
    except Exception:
        pval = np.nan

    return float(b), float(pval), resid


def half_life(spread: pd.Series) -> float:
    """
    Half-life of mean reversion via: Δs_t = a + b*s_{t-1}
    half-life = -ln(2)/b if b < 0 else inf
    """
    s = spread.dropna()
    if len(s) < 50:
        return float("nan")

    s_lag = s.shift(1).dropna()
    ds = (s - s.shift(1)).dropna()
    df = pd.concat([ds.rename("ds"), s_lag.rename("s1")], axis=1).dropna()
    if len(df) < 30:
        return float("nan")

    X = sm.add_constant(df["s1"].values)
    model = sm.OLS(df["ds"].values, X).fit()
    b = float(model.params[1])

    if not np.isfinite(b) or b >= 0:
        return float("inf")
    return float(-math.log(2) / b)


def score_pair(corr_val: float, pval: float, hl: float, cfg: Config) -> float:
    """
    Higher score = better.
    """
    if not np.isfinite(corr_val):
        corr_val = 0.0
    if not np.isfinite(pval):
        pval = 1.0
    if not np.isfinite(hl):
        hl = float("inf")

    logp = -math.log10(max(pval, 1e-300))
    hl_term = 0.0 if (hl == float("inf")) else (1.0 / (hl + 1.0))

    return cfg.W_CORR * corr_val + cfg.W_LOGP * logp + cfg.W_HALF_LIFE * hl_term


# =============================================================================
# MAIN
# =============================================================================
def main(cfg: Config) -> None:
    universe = pd.read_csv(cfg.UNIVERSE_CSV)
    if not {"ticker", "sector"}.issubset(set(universe.columns)):
        raise ValueError("universe.csv must contain at least columns: ticker, sector")

    # 1) pick top tickers per sector
    top_by_sector = pick_top_per_sector(universe, cfg)
    all_top_tickers = sorted(set(itertools.chain.from_iterable(top_by_sector.values())))

    print(f"[INFO] Sectors with >=2 tickers after top-N: {len(top_by_sector)}")
    print(f"[INFO] Total unique tickers to download: {len(all_top_tickers)}")

    # 2) download prices once
    close, _ = fetch_prices_and_volume(all_top_tickers, cfg.START, cfg.END, auto_adjust=cfg.AUTO_ADJUST)
    if close.empty:
        raise RuntimeError("No price data downloaded. Check tickers / network / dates.")

    # log-prices
    close = close.replace(0.0, np.nan).dropna(how="all")
    logp_all = np.log(close)

    all_rows = []

    # 3) for each sector: corr screen -> cointegration -> rank
    for sector, ticks in top_by_sector.items():
        ticks = [t for t in ticks if t in logp_all.columns]
        if len(ticks) < 2:
            continue

        lp_sec = logp_all[ticks].dropna(axis=1, how="any")
        if lp_sec.shape[1] < 2:
            continue

        corr_pairs = rolling_corr_pairs(
            lp_sec, lookback=cfg.CORR_LOOKBACK_DAYS,
            corr_thresh=cfg.CORR_THRESHOLD,
            max_pairs=cfg.MAX_CORR_PAIRS_PER_SECTOR
        )

        for a, b, c in corr_pairs:
            # OLS/ADF (Engle-Granger style)
            beta, pval, resid = engle_granger_ols_adf(log_y=lp_sec[a], log_x=lp_sec[b])
            hl = half_life(resid) if len(resid) else float("nan")

            row = {
                "sector": sector,
                "ticker_y": a,
                "ticker_x": b,
                "corr": c,
                "hedge_beta": beta,
                "adf_pvalue": pval,
                "half_life": hl,
            }
            row["score"] = score_pair(c, pval, hl, cfg)
            row["cointegrated"] = (np.isfinite(pval) and pval <= cfg.ADF_PVALUE_MAX)
            all_rows.append(row)

    if not all_rows:
        print("[WARN] No candidate pairs found. Try lowering CORR_THRESHOLD or increasing TOP_N_PER_SECTOR.")
        return

    df_all = pd.DataFrame(all_rows)

    # Prefer cointegrated pairs first; then by score
    df_all = df_all.sort_values(by=["sector", "cointegrated", "score"], ascending=[True, False, False])

    # Best per sector (cointegrated if possible; otherwise highest score)
    best_rows = []
    for sector, g in df_all.groupby("sector"):
        g = g.copy()
        g1 = g[g["cointegrated"] == True]
        pick = g1.iloc[0] if len(g1) else g.iloc[0]
        best_rows.append(pick)

    df_best = pd.DataFrame(best_rows).sort_values(by=["cointegrated", "score"], ascending=[False, False])

    # Save
    df_all.to_csv("sector_all_candidates.csv", index=False)
    df_best.to_csv("sector_best_pairs.csv", index=False)

    # Print summary
    print("\n=== BEST PAIR PER SECTOR ===")
    print(df_best[["sector", "ticker_y", "ticker_x", "corr", "adf_pvalue", "half_life", "score", "cointegrated"]].to_string(index=False))

    print("\n[OK] Wrote:")
    print(" - sector_all_candidates.csv")
    print(" - sector_best_pairs.csv")


if __name__ == "__main__":
    main(CFG)

FileNotFoundError: [Errno 2] No such file or directory: 'universe.csv'

In [1]:
import yfinance as yf
import pandas as pd
import time

def create_sectoral_universe(n_top=5):
    # 1. Define your initial Ticker List (Example: Nifty 50 Tickers)
    # For a full universe, you'd typically import a CSV of Nifty 500 tickers from the NSE website.
    # Note: Indian tickers in yfinance must end with '.NS'
    base_tickers = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS", 
                    "HINDUNILVR.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "LICI.NS"] 
    
    universe_data = []

    print(f"Fetching data for {len(base_tickers)} companies...")

    for ticker_symbol in base_tickers:
        try:
            ticker = yf.Ticker(ticker_symbol)
            info = ticker.info
            
            # Extract required fields
            sector = info.get('sector', 'Unknown')
            total_market_cap = info.get('marketCap', 0)
            float_shares = info.get('floatShares', 0)
            current_price = info.get('currentPrice', 0)
            
            # Calculate Free Float Market Cap
            # If floatShares isn't available, we fallback to total market cap as an estimate
            free_float_mcap = current_price * float_shares if float_shares else total_market_cap

            universe_data.append({
                'ticker': ticker_symbol,
                'sector': sector,
                'free_float_market_cap': free_float_mcap
            })
            
            # Ethical scraping: small delay to avoid rate limits
            time.sleep(0.1) 
            
        except Exception as e:
            print(f"Could not fetch {ticker_symbol}: {e}")

    # 2. Process Data
    df = pd.DataFrame(universe_data)
    
    # 3. Filter Top N per Sector
    # We sort by market cap within each group and take the top N
    universe_csv = (
        df.sort_values(['sector', 'free_float_market_cap'], ascending=[True, False])
          .groupby('sector')
          .head(n_top)
    )

    # 4. Save to CSV
    universe_csv.to_csv('universe.csv', index=False)
    print("Success! universe.csv created.")
    return universe_csv

# Execute
top_n_universe = create_sectoral_universe(n_top=2)
print(top_n_universe)

Fetching data for 10 companies...
Success! universe.csv created.
          ticker                  sector  free_float_market_cap
8  BHARTIARTL.NS  Communication Services           5.278952e+12
6         ITC.NS      Consumer Defensive           3.010640e+12
5  HINDUNILVR.NS      Consumer Defensive           2.429726e+12
0    RELIANCE.NS                  Energy           9.293872e+12
2    HDFCBANK.NS      Financial Services           1.378092e+13
4   ICICIBANK.NS      Financial Services           1.000640e+13
3        INFY.NS              Technology           4.439725e+12
1         TCS.NS              Technology           2.701642e+12
